# Module 1, Video 3: Tuning Decoding Parameters in Practice

## Understanding and Controlling Generative AI Model Outputs

**Learning Objectives:**
- Understand how generative AI models produce text from probabilities to words
- Explore the three core generation parameters: **Temperature**, **Top-k**, and **Top-p**
- Experiment with different parameter combinations for various use cases
- Learn to balance creativity vs. consistency in model outputs

---

## Real-World Scenario: The AI Marketing Assistant

Imagine you're working at a marketing agency where the AI assistant generates content for different purposes:
- **Creative blog posts** (need variety and creativity)
- **Technical summaries** (need accuracy and consistency)
- **Customer support responses** (need clarity and reliability)

The challenge: The outputs are either too boring (repetitive, predictable) or too random (inconsistent, off-topic). 

**Your mission:** Learn to tune the decoding parameters to control the model's creativity and consistency!

## Setup: Installing Required Libraries

We'll use the Hugging Face Transformers library to load a base text generation model and experiment with its parameters.

In [1]:
# Install required libraries
!pip install transformers torch accelerate -q

In [3]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")
print(f"Using PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Libraries imported successfully!
Using PyTorch version: 2.9.0
CUDA available: False


## Part 1: Loading a Base Language Model

We'll use GPT-2 as our base model for this demonstration. It's small enough to run on most systems but powerful enough to demonstrate decoding parameters.

In [4]:
# Load model and tokenizer
model_name = "gpt2"  # Using GPT-2 for demonstration

print(f"Loading {model_name} model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading gpt2 model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

W1111 10:47:43.070000 50694 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Model loaded successfully!
Model parameters: 124,439,808


## Part 2: Understanding How Models Generate Text

### From Probabilities to Words

Language models predict the next word by:
1. Converting input text to tokens
2. Computing probability distributions over all possible next tokens
3. Selecting the next token based on decoding parameters
4. Repeating until reaching the desired length or stop token

Let's visualize this process!

In [36]:
# Helper function to generate text with custom parameters
def generate_text(prompt, temperature=1.0, top_k=50, top_p=1.0, repetition_penalty=1.0, max_length=100, num_return_sequences=1):
    """
    Generate text using specified decoding parameters.
    
    Args:
        prompt: Input text to continue
        temperature: Controls randomness (0.1-2.0)
        top_k: Limits vocabulary to top k tokens
        top_p: Nucleus sampling threshold
        repetition_penalty: Penalizes token repetition (1.0 = no penalty)
        max_length: Maximum output length in tokens
        num_return_sequences: Number of different outputs to generate
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        num_return_sequences=num_return_sequences,
        do_sample=True,  # Enable sampling
        pad_token_id=tokenizer.eos_token_id
    )
    
    results = []
    for i, output in enumerate(outputs):
        generated_text = tokenizer.decode(output, skip_special_tokens=True)
        results.append(generated_text)
    
    return results

print(" Generation function ready!")

 Generation function ready!


## Part 3: The Five Key Decoding Parameters

Let's explore the most important parameters for controlling LLM output:

### Parameter 1: Temperature

**What it does:** Controls the randomness of predictions by scaling the probability distribution.

- **Low temperature (0.0-0.3):** Makes the model more deterministic
  - Selects high-probability tokens
  - Output is consistent and predictable
  - Best for: factual content, technical writing, customer support

- **Medium temperature (0.7-1.0):** Balanced creativity and coherence
  - Default setting for most applications
  - Best for: general content generation

- **High temperature (1.2-2.0):** Increases randomness and creativity
  - Explores less probable tokens
  - Output is diverse and surprising
  - Best for: creative writing, brainstorming, varied responses

In [32]:
# Experiment 1: Testing Different Temperatures
prompt = "Artificial intelligence is transforming"

print("=" * 80)
print("EXPERIMENT 1: Temperature Comparison")
print("=" * 80)
print(f"\nPrompt: '{prompt}'\n")

temperatures = [0.3, 0.7, 1.2]

for temp in temperatures:
    print(f"\n{'─' * 80}")
    print(f" Temperature = {temp}")
    print(f"{'─' * 80}")
    
    results = generate_text(
        prompt, 
        temperature=temp, 
        top_k=50, 
        top_p=1.0,
        max_length=60,
        num_return_sequences=2
    )
    
    for i, text in enumerate(results, 1):
        print(f"\nOutput {i}:")
        print(text)

print("\n" + "=" * 80)
print(" Observation: Notice how lower temperatures produce more similar, focused outputs,")
print("   while higher temperatures create more diverse and creative variations!")
print("=" * 80)

EXPERIMENT 1: Temperature Comparison

Prompt: 'Artificial intelligence is transforming'


────────────────────────────────────────────────────────────────────────────────
 Temperature = 0.3
────────────────────────────────────────────────────────────────────────────────

Output 1:
Artificial intelligence is transforming the way we think about our world.

The world of artificial intelligence is transforming the way we think about our world.

The world of artificial intelligence is transforming the way we think about our world.

The world of artificial intelligence is transforming the way we think

Output 2:
Artificial intelligence is transforming the world. It's a process that is accelerating, and it's accelerating faster than any other technology.

In the past, we've seen the emergence of AI as a way of making decisions. We have to make decisions, and we have to make decisions that are

────────────────────────────────────────────────────────────────────────────────
 Temperature = 0.7


### Parameter 2: Top-k Sampling

**What it does:** Limits the model's vocabulary to the k most likely next tokens.

- **Small k (10-20):** Very restrictive
  - Only considers most probable words
  - Highly focused and consistent output
  - Risk: can be repetitive

- **Medium k (40-50):** Balanced approach (default)
  - Good variety while maintaining coherence

- **Large k (100+):** More exploratory
  - Allows less common word choices
  - More diverse outputs
  - Risk: can be less coherent

In [33]:
# Experiment 2: Testing Different Top-k Values
prompt = "The future of renewable energy includes"

print("=" * 80)
print("EXPERIMENT 2: Top-k Comparison")
print("=" * 80)
print(f"\nPrompt: '{prompt}'\n")

top_k_values = [10, 50, 100]

for k in top_k_values:
    print(f"\n{'─' * 80}")
    print(f" Top-k = {k}")
    print(f"{'─' * 80}")
    
    results = generate_text(
        prompt, 
        temperature=0.8, 
        top_k=k, 
        top_p=1.0,
        max_length=60,
        num_return_sequences=2
    )
    
    for i, text in enumerate(results, 1):
        print(f"\nOutput {i}:")
        print(text)

print("\n" + "=" * 80)
print(" Observation: Lower top-k values produce more focused vocabulary,")
print("   while higher values allow for more word variety!")
print("=" * 80)

EXPERIMENT 2: Top-k Comparison

Prompt: 'The future of renewable energy includes'


────────────────────────────────────────────────────────────────────────────────
 Top-k = 10
────────────────────────────────────────────────────────────────────────────────

Output 1:
The future of renewable energy includes the development and deployment of wind and solar energy. We need to continue to grow renewables and reduce our dependence on fossil fuels and we need to continue to support our renewable energy customers," said Bill Condon, President of the United States Energy Alliance.

"Today's

Output 2:
The future of renewable energy includes a transition in where we will use less fossil fuels, and a transition to lower carbon emissions.

"This is an opportunity for the world to see what we can do to reduce the amount of fossil fuels we have on the planet," said Mr Burdick.

────────────────────────────────────────────────────────────────────────────────
 Top-k = 50
────────────────────────────

### Parameter 3: Top-p (Nucleus Sampling)

**What it does:** Dynamically selects tokens whose cumulative probability reaches threshold p.

- **Low p (0.5-0.7):** Conservative
  - Only uses highest probability tokens
  - Consistent, safe outputs
  - Vocabulary size adapts to context

- **Medium p (0.85-0.9):** Balanced (common default)
  - Good mix of reliability and variety

- **High p (0.95-1.0):** Exploratory
  - Considers wider range of possibilities
  - More creative and diverse
  - May include less common phrases

**Key advantage over top-k:** Adapts vocabulary size based on probability distribution!

###  Parameter 4: Repetition Penalty

**What it does:** Discourages the model from repeating the same words or phrases.

- **No penalty (1.0):** Default behavior, no penalty applied
  - Model may naturally repeat for emphasis
  - Can lead to repetitive outputs in longer texts

- **Light penalty (1.1-1.3):** Gentle discouragement
  - Reduces obvious repetition
  - Maintains natural language flow
  - Best for: most content generation tasks

- **Strong penalty (1.5-2.0):** Aggressive repetition avoidance
  - Forces vocabulary diversity
  - Can make text feel unnatural
  - Best for: creative tasks requiring maximum variety

**When to use:** Essential for longer outputs where repetition becomes problematic!

### Parameter 5: Max Length

**What it does:** Controls the number of tokens generated in the output.

- **Short (20-50 tokens):** Brief responses
  - Quick answers, concise summaries
  - Best for: Q&A, short descriptions

- **Medium (50-150 tokens):** Paragraph-length responses
  - Detailed explanations
  - Best for: documentation, blog paragraphs

- **Long (200+ tokens):** Extended content
  - Full articles, comprehensive answers
  - Best for: long-form content, detailed analysis

**Important:** Longer outputs may require repetition penalty adjustment!

In [34]:
# Experiment 3: Testing Different Top-p Values
prompt = "Machine learning algorithms can help us"

print("=" * 80)
print("EXPERIMENT 3: Top-p (Nucleus Sampling) Comparison")
print("=" * 80)
print(f"\nPrompt: '{prompt}'\n")

top_p_values = [0.5, 0.8, 0.95]

for p in top_p_values:
    print(f"\n{'─' * 80}")
    print(f" Top-p = {p}")
    print(f"{'─' * 80}")
    
    results = generate_text(
        prompt, 
        temperature=0.8, 
        top_k=0,  # Disable top-k to isolate top-p effect
        top_p=p,
        max_length=60,
        num_return_sequences=2
    )
    
    for i, text in enumerate(results, 1):
        print(f"\nOutput {i}:")
        print(text)

print("\n" + "=" * 80)
print(" Observation: Top-p dynamically adjusts the vocabulary size,")
print("   providing more nuanced control than fixed top-k!")
print("=" * 80)

EXPERIMENT 3: Top-p (Nucleus Sampling) Comparison

Prompt: 'Machine learning algorithms can help us'


────────────────────────────────────────────────────────────────────────────────
 Top-p = 0.5
────────────────────────────────────────────────────────────────────────────────

Output 1:
Machine learning algorithms can help us predict and correct for any of the following:

Non-linear learning

Natural language processing

Non-linear programming

Non-linear learning

Non-linear programming

Non-linear programming

Non-linear programming

Non-

Output 2:
Machine learning algorithms can help us to understand and predict the future.

"We're looking at a lot of different ways to learn about the future," said Eric Knapp, an associate professor of computer science at the University of Pennsylvania. "The most exciting thing is that we can use these algorithms

────────────────────────────────────────────────────────────────────────────────
 Top-p = 0.8
─────────────────────────────────────────

###  Experiment 3b: Testing Repetition Penalty

Repetition penalty is especially important for longer text generation. Let's see how it affects output!

In [35]:
# Experiment 3b: Testing Repetition Penalty
prompt = "The key benefits of artificial intelligence include"

print("=" * 80)
print("EXPERIMENT 3b: Repetition Penalty Comparison")
print("=" * 80)
print(f"\nPrompt: '{prompt}'\n")
print("Note: Generating longer outputs (150 tokens) to see repetition effects\n")

repetition_penalties = [1.0, 1.2, 1.5]

for penalty in repetition_penalties:
    print(f"\n{'─' * 80}")
    print(f" Repetition Penalty = {penalty}")
    if penalty == 1.0:
        print("   (No penalty - natural model behavior)")
    elif penalty < 1.3:
        print("   (Light penalty - gentle discouragement)")
    else:
        print("   (Strong penalty - aggressive diversity)")
    print(f"{'─' * 80}")
    
    results = generate_text(
        prompt, 
        temperature=0.7, 
        top_k=50, 
        top_p=0.9,
        repetition_penalty=penalty,
        max_length=150,  # Longer to see repetition
        num_return_sequences=1
    )
    
    print(f"\nOutput:")
    print(results[0])

print("\n" + "=" * 80)
print(" Observation: Higher repetition penalty forces more vocabulary diversity,")
print("   especially important for longer text generation!")
print("=" * 80)

EXPERIMENT 3b: Repetition Penalty Comparison

Prompt: 'The key benefits of artificial intelligence include'

Note: Generating longer outputs (150 tokens) to see repetition effects


────────────────────────────────────────────────────────────────────────────────
 Repetition Penalty = 1.0
   (No penalty - natural model behavior)
────────────────────────────────────────────────────────────────────────────────

Output:
The key benefits of artificial intelligence include:

It can learn more quickly than humans. It can read the mind of a person. It can learn a lot about the human mind. It can see more clearly than human minds can. It can see more clearly than human minds can.

It can learn more quickly than humans. It can read the mind of a person. It can learn a lot about the human mind. It can see more clearly than human minds can. It can see more clearly than human minds can. It can also learn more quickly than humans. It can learn a lot more quickly than humans.

It can learn more quick

## Part 4: Comparing Decoding Strategies Side-by-Side

Let's compare different decoding strategies:
- **Greedy Decoding:** Always picks most likely token (deterministic)
- **Beam Search:** Explores multiple paths simultaneously
- **Sampling with parameters:** Our tunable approach

In [37]:
# Experiment 4: Comparing Decoding Strategies
prompt = "The benefits of sustainable development are"

print("=" * 80)
print("EXPERIMENT 4: Beam Search vs. Sampling Strategies")
print("=" * 80)
print(f"\nPrompt: '{prompt}'\n")

# Greedy (deterministic - temperature not used, do_sample=False)
print("\n" + "─" * 80)
print(" Strategy: GREEDY DECODING (Deterministic)")
print("Parameters: Always picks most likely token")
print("─" * 80)
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Beam search
print("\n" + "─" * 80)
print(" Strategy: BEAM SEARCH (num_beams=5)")
print("Parameters: Explores 5 paths simultaneously")
print("─" * 80)
outputs = model.generate(**inputs, max_length=60, num_beams=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Stochastic with temperature
print("\n" + "─" * 80)
print(" Strategy: SAMPLING (Temperature=0.7, Top-p=0.9)")
print("Parameters: Balanced creativity and consistency")
print("─" * 80)
results = generate_text(prompt, temperature=0.7, top_k=50, top_p=0.9, max_length=60, num_return_sequences=1)
print(results[0])

print("\n" + "=" * 80)
print(" Key Insight: Beam search is deterministic and finds 'safe' outputs,")
print("   while sampling with temperature allows for creative exploration!")
print("=" * 80)

EXPERIMENT 4: Beam Search vs. Sampling Strategies

Prompt: 'The benefits of sustainable development are'


────────────────────────────────────────────────────────────────────────────────
 Strategy: GREEDY DECODING (Deterministic)
Parameters: Always picks most likely token
────────────────────────────────────────────────────────────────────────────────
The benefits of sustainable development are clear. The world's population is growing at a faster rate than the rate of population growth in the past. The world's population is growing at a faster rate than the rate of population growth in the past. The world's population is growing at a faster rate than the

────────────────────────────────────────────────────────────────────────────────
 Strategy: BEAM SEARCH (num_beams=5)
Parameters: Explores 5 paths simultaneously
────────────────────────────────────────────────────────────────────────────────
The benefits of sustainable development are obvious, but there are also some drawbacks.

Fir

## Part 4b: Systematic Parameter Exploration & A/B Testing

In production environments, you need a systematic approach to find optimal parameters. Let's learn how to:
1. **Define test dimensions** - What parameters to vary
2. **Create parameter combinations** - Build a test matrix
3. **Run A/B tests** - Compare outputs systematically
4. **Evaluate and select** - Choose the best configuration

### The A/B Testing Framework for LLM Parameters

In [38]:
# A/B Testing Framework
import itertools
from typing import List, Dict, Any

def ab_test_parameters(prompt: str, parameter_grid: Dict[str, List[Any]], 
                       max_length: int = 80, num_samples: int = 1):
    """
    Systematically test different parameter combinations.
    
    Args:
        prompt: The input prompt to test
        parameter_grid: Dictionary of parameter names to lists of values to test
        max_length: Maximum output length
        num_samples: Number of samples per configuration
    
    Returns:
        List of results with parameter configs and outputs
    """
    results = []
    
    # Generate all combinations
    param_names = list(parameter_grid.keys())
    param_values = list(parameter_grid.values())
    
    for combination in itertools.product(*param_values):
        config = dict(zip(param_names, combination))
        
        # Generate text with this configuration
        outputs = generate_text(
            prompt,
            temperature=config.get('temperature', 1.0),
            top_k=config.get('top_k', 50),
            top_p=config.get('top_p', 1.0),
            repetition_penalty=config.get('repetition_penalty', 1.0),
            max_length=max_length,
            num_return_sequences=num_samples
        )
        
        results.append({
            'config': config,
            'outputs': outputs
        })
    
    return results

def display_ab_test_results(results: List[Dict], prompt: str):
    """
    Display A/B test results in a readable format.
    """
    print("=" * 80)
    print("A/B TESTING RESULTS")
    print("=" * 80)
    print(f"\nPrompt: '{prompt}'\n")
    
    for i, result in enumerate(results, 1):
        config = result['config']
        outputs = result['outputs']
        
        print(f"\n{'─' * 80}")
        print(f"Test Configuration {i}")
        print(f"{'─' * 80}")
        
        # Display configuration
        for param, value in config.items():
            print(f"  {param}: {value}")
        
        # Display outputs
        for j, output in enumerate(outputs, 1):
            print(f"\n  Output {j}:")
            print(f"  {output}")
    
    print("\n" + "=" * 80)

print(" A/B testing framework ready!")

 A/B testing framework ready!


### A/B Test Example 1: Finding Optimal Temperature

Let's test different temperature values while keeping other parameters constant.

In [31]:
# A/B Test: Temperature variations
prompt = "The future of sustainable technology will"

# Define parameter grid - testing only temperature
parameter_grid = {
    'temperature': [0.3, 0.7, 1.0],
    'top_k': [50],  # Keep constant
    'top_p': [0.9],  # Keep constant
    'repetition_penalty': [1.0]  # Keep constant
}

# Run A/B test
results = ab_test_parameters(prompt, parameter_grid, max_length=70, num_samples=2)

# Display results
display_ab_test_results(results, prompt)

print("\n Analysis: Compare consistency (low temp) vs. creativity (high temp)")
print("   Which temperature best suits your use case?")

A/B TESTING RESULTS

Prompt: 'The future of sustainable technology will'


────────────────────────────────────────────────────────────────────────────────
Test Configuration 1
────────────────────────────────────────────────────────────────────────────────
  temperature: 0.3
  top_k: 50
  top_p: 0.9
  repetition_penalty: 1.0

  Output 1:
  The future of sustainable technology will be determined by the success of the next generation of smart devices and the future of the Internet of Things.

The future of sustainable technology will be determined by the success of the next generation of smart devices and the future of the Internet of Things.

The future of sustainable technology will be determined by the success of

  Output 2:
  The future of sustainable technology will depend on the success of the next generation of smart cars, and the future of the future of the future of the future of the future of the future of the future of the future of the future of the future of the future of 

### A/B Test Example 2: Multi-Parameter Optimization

Now let's test combinations of multiple parameters to find the optimal configuration.

In [30]:
# A/B Test: Multiple parameter combinations
prompt = "Best practices for AI model deployment include"

# Define parameter grid - testing temperature and top-p combinations
parameter_grid = {
    'temperature': [0.3, 0.8],  # Low vs medium-high
    'top_k': [50],  # Keep constant
    'top_p': [0.7, 0.95],  # Conservative vs exploratory
    'repetition_penalty': [1.0]  # Keep constant
}

print("Testing 4 configurations (2 temps × 2 top-p values)...\n")

# Run A/B test
results = ab_test_parameters(prompt, parameter_grid, max_length=70, num_samples=1)

# Display results
display_ab_test_results(results, prompt)

print("\n Analysis: Look for the 'sweet spot' that balances your needs:")
print("   • (temp=0.3, top_p=0.7) → Maximum consistency")
print("   • (temp=0.3, top_p=0.95) → Consistent but varied vocabulary")
print("   • (temp=0.8, top_p=0.7) → Creative but focused")
print("   • (temp=0.8, top_p=0.95) → Maximum creativity and diversity")

Testing 4 configurations (2 temps × 2 top-p values)...

A/B TESTING RESULTS

Prompt: 'Best practices for AI model deployment include'


────────────────────────────────────────────────────────────────────────────────
Test Configuration 1
────────────────────────────────────────────────────────────────────────────────
  temperature: 0.3
  top_k: 50
  top_p: 0.7
  repetition_penalty: 1.0

  Output 1:
  Best practices for AI model deployment include:

- Automatically deploy to a new database

- Automatically deploy to a new database - Automatically deploy to a new database

- Automatically deploy to a new database - Automatically deploy to a new database

- Automatically deploy to a new database - Automatically deploy to a

────────────────────────────────────────────────────────────────────────────────
Test Configuration 2
────────────────────────────────────────────────────────────────────────────────
  temperature: 0.3
  top_k: 50
  top_p: 0.95
  repetition_penalty: 1.0

  Output 1:
  

### Systematic Parameter Space Exploration

For production systems, you want to explore the parameter space systematically. Here's a comprehensive test:

In [28]:
# Comprehensive parameter exploration
def explore_parameter_space(prompt: str, use_case: str = "general"):
    """
    Explore parameter space with predefined configurations for different use cases.
    
    Args:
        prompt: Input prompt to test
        use_case: 'creative', 'technical', 'balanced', or 'general'
    """
    
    # Define configurations for different use cases
    configurations = {
        'creative': {
            'A - High Creativity': {'temperature': 1.0, 'top_k': 80, 'top_p': 0.95, 'repetition_penalty': 1.2},
            'B - Moderate Creativity': {'temperature': 0.8, 'top_k': 60, 'top_p': 0.9, 'repetition_penalty': 1.1},
            'C - Balanced': {'temperature': 0.7, 'top_k': 50, 'top_p': 0.85, 'repetition_penalty': 1.0}
        },
        'technical': {
            'A - Maximum Precision': {'temperature': 0.2, 'top_k': 10, 'top_p': 0.6, 'repetition_penalty': 1.0},
            'B - Moderate Precision': {'temperature': 0.4, 'top_k': 30, 'top_p': 0.75, 'repetition_penalty': 1.1},
            'C - Balanced': {'temperature': 0.5, 'top_k': 40, 'top_p': 0.8, 'repetition_penalty': 1.0}
        },
        'general': {
            'A - Conservative': {'temperature': 0.5, 'top_k': 30, 'top_p': 0.75, 'repetition_penalty': 1.0},
            'B - Balanced': {'temperature': 0.7, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.1},
            'C - Creative': {'temperature': 0.9, 'top_k': 70, 'top_p': 0.95, 'repetition_penalty': 1.2}
        }
    }
    
    configs = configurations.get(use_case, configurations['general'])
    
    print("=" * 80)
    print(f"PARAMETER SPACE EXPLORATION: {use_case.upper()} Use Case")
    print("=" * 80)
    print(f"\nPrompt: '{prompt}'\n")
    
    for name, config in configs.items():
        print(f"\n{'─' * 80}")
        print(f"Configuration: {name}")
        print(f"{'─' * 80}")
        print(f"Parameters: {config}\n")
        
        outputs = generate_text(
            prompt,
            temperature=config['temperature'],
            top_k=config['top_k'],
            top_p=config['top_p'],
            repetition_penalty=config['repetition_penalty'],
            max_length=80,
            num_return_sequences=2
        )
        
        for i, output in enumerate(outputs, 1):
            print(f"Sample {i}: {output}\n")
    
    print("=" * 80)
    print(" Next Steps:")
    print("   1. Evaluate outputs for quality, relevance, and creativity")
    print("   2. Select the configuration that best meets your requirements")
    print("   3. Fine-tune the chosen configuration if needed")
    print("   4. Test with multiple prompts to ensure consistency")
    print("=" * 80)

print(" Parameter space exploration function ready!")

 Parameter space exploration function ready!


In [29]:
# Example: Explore parameter space for creative writing
explore_parameter_space(
    prompt="Write a compelling opening for a science fiction story:",
    use_case="creative"
)

PARAMETER SPACE EXPLORATION: CREATIVE Use Case

Prompt: 'Write a compelling opening for a science fiction story:'


────────────────────────────────────────────────────────────────────────────────
Configuration: A - High Creativity
────────────────────────────────────────────────────────────────────────────────
Parameters: {'temperature': 1.0, 'top_k': 80, 'top_p': 0.95, 'repetition_penalty': 1.2}

Sample 1: Write a compelling opening for a science fiction story: It takes place over 40 years, but at the beginning of every decade this time I wonder if there is an age when we will be able to read other people's stories as well. For example, what would happen here in our 20s after you write? What happens around 200 books about war that have absolutely no resemblance to reality and can't even

Sample 2: Write a compelling opening for a science fiction story: A woman who gets the best from her husband, learns an important lesson that can save life at some point (usually in "Hollow Stone".);

In [16]:
# Example: Explore parameter space for technical content
explore_parameter_space(
    prompt="Explain the concept of gradient descent in machine learning:",
    use_case="technical"
)

PARAMETER SPACE EXPLORATION: TECHNICAL Use Case

Prompt: 'Explain the concept of gradient descent in machine learning:'


────────────────────────────────────────────────────────────────────────────────
Configuration: A - Maximum Precision
────────────────────────────────────────────────────────────────────────────────
Parameters: {'temperature': 0.2, 'top_k': 10, 'top_p': 0.6, 'repetition_penalty': 1.0}

Sample 1: Explain the concept of gradient descent in machine learning:

The gradient descent algorithm is a simple, yet powerful, way to learn a neural network. It is based on the idea that the neural network is a collection of neurons that are connected to a network of neurons. The network is a collection of neurons that are connected to a network of neurons. The network is a collection of neurons that are

Sample 2: Explain the concept of gradient descent in machine learning:

The gradient descent algorithm is a simple, yet powerful, way to learn a neural network. It is a simple, ye

## Part 5: Real-World Application Scenarios

Let's apply what we've learned to solve the marketing agency's challenges!

In [27]:
# Scenario 1: Creative Blog Post (High Variability)
print("=" * 80)
print("SCENARIO 1: Creative Blog Introduction")
print("Goal: Generate engaging, creative content with variety")
print("=" * 80)

blog_prompt = "5 innovative ways to use AI in your daily life:"

print(f"\nPrompt: '{blog_prompt}'\n")
print("Parameters:  Temperature=1.0,  Top-k=50,  Top-p=0.95")
print("Strategy: High creativity for blog content\n")

blog_results = generate_text(
    blog_prompt,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    max_length=100,
    num_return_sequences=2
)

for i, text in enumerate(blog_results, 1):
    print(f"\n{'─' * 80}")
    print(f"Creative Variation {i}:")
    print(f"{'─' * 80}")
    print(text)

print("\n Result: Diverse, engaging content perfect for blog posts!\n")

SCENARIO 1: Creative Blog Introduction
Goal: Generate engaging, creative content with variety

Prompt: '5 innovative ways to use AI in your daily life:'

Parameters: 🌡️ Temperature=1.0, 🎯 Top-k=50, 🎲 Top-p=0.95
Strategy: High creativity for blog content


────────────────────────────────────────────────────────────────────────────────
Creative Variation 1:
────────────────────────────────────────────────────────────────────────────────
5 innovative ways to use AI in your daily life:

In our previous blog post, we talked about The Brain's Aptitude Test, which aims to determine how people will assess what works and what doesn't when using AI. Using a simple test called the BOLD paradigm for training new brain cells, we learned that the main effect of using AI systems is to improve individual brain performance. We then showed that training new neurons with Aptitude, while retaining previous performance over time, increases

─────────────────────────────────────────────────────────────────

In [26]:
# Scenario 2: Technical Summary (Low Variability, High Precision)
print("=" * 80)
print("SCENARIO 2: Technical Documentation")
print("Goal: Generate accurate, consistent technical content")
print("=" * 80)

tech_prompt = "The process of model fine-tuning involves"

print(f"\nPrompt: '{tech_prompt}'\n")
print("Parameters:  Temperature=0.3,  Top-k=20,  Top-p=0.7")
print("Strategy: Low temperature for precision and consistency\n")

tech_results = generate_text(
    tech_prompt,
    temperature=0.3,
    top_k=20,
    top_p=0.7,
    max_length=100,
    num_return_sequences=2
)

for i, text in enumerate(tech_results, 1):
    print(f"\n{'─' * 80}")
    print(f"Technical Output {i}:")
    print(f"{'─' * 80}")
    print(text)

print("\n Result: Consistent, reliable technical explanations!\n")

SCENARIO 2: Technical Documentation
Goal: Generate accurate, consistent technical content

Prompt: 'The process of model fine-tuning involves'

Parameters:  Temperature=0.3,  Top-k=20,  Top-p=0.7
Strategy: Low temperature for precision and consistency


────────────────────────────────────────────────────────────────────────────────
Technical Output 1:
────────────────────────────────────────────────────────────────────────────────
The process of model fine-tuning involves a series of steps. The first step is to determine the optimal model for the model. The second step is to determine the optimal model for the model's parameters. The third step is to determine the optimal model for the model's parameters. The fourth step is to determine the optimal model for the model's parameters. The fifth step is to determine the optimal model for the model's parameters. The sixth step is to determine the optimal model for the model's

───────────────────────────────────────────────────────────────

In [25]:
# Scenario 3: Customer Support (Balanced - Clear and Helpful)
print("=" * 80)
print("SCENARIO 3: Customer Support Response")
print("Goal: Generate clear, helpful, on-topic responses")
print("=" * 80)

support_prompt = "Thank you for contacting our support team. To resolve your issue,"

print(f"\nPrompt: '{support_prompt}'\n")
print("Parameters: Temperature=0.5,  Top-k=40,  Top-p=0.8")
print("Strategy: Balanced approach for helpful, clear responses\n")

support_results = generate_text(
    support_prompt,
    temperature=0.5,
    top_k=40,
    top_p=0.8,
    max_length=100,
    num_return_sequences=2
)

for i, text in enumerate(support_results, 1):
    print(f"\n{'─' * 80}")
    print(f"Support Response {i}:")
    print(f"{'─' * 80}")
    print(text)

print("\n Result: Clear, helpful responses that stay on-topic!\n")

SCENARIO 3: Customer Support Response
Goal: Generate clear, helpful, on-topic responses

Prompt: 'Thank you for contacting our support team. To resolve your issue,'

Parameters: Temperature=0.5,  Top-k=40,  Top-p=0.8
Strategy: Balanced approach for helpful, clear responses


────────────────────────────────────────────────────────────────────────────────
Support Response 1:
────────────────────────────────────────────────────────────────────────────────
Thank you for contacting our support team. To resolve your issue, please contact us at support@mw.com.

We're sorry, currently this live video stream is only available inside of Utah or an approved RSL broadcast territory. We base your location on your IP address. Some providers IP addresses may show your location outside of the state, even though you are physically within the state boundaries. For more information about RSL on KSL, please see our FAQ.

Photos


───────────────────────────────────────────────────────────────────────────

## Part 6: Parameter Tuning Guidelines - Quick Reference

Here's a comprehensive decision framework for choosing the right parameters:

| **Use Case** | **Temperature** | **Top-k** | **Top-p** | **Rep. Penalty** | **Max Length** | **Why?** |
|--------------|-----------------|-----------|-----------|------------------|----------------|----------|
| **Creative writing** | 0.8-1.2 | 50-100 | 0.9-0.95 | 1.1-1.3 | 100-300 | High variability, diverse expressions |
| **Technical docs** | 0.2-0.4 | 10-30 | 0.6-0.75 | 1.0-1.1 | 50-200 | Precision, consistency, factual |
| **Customer support** | 0.4-0.6 | 30-50 | 0.75-0.85 | 1.0-1.1 | 50-150 | Balanced clarity and helpfulness |
| **Code generation** | 0.1-0.3 | 5-20 | 0.5-0.7 | 1.0 | 100-500 | Deterministic, syntactically correct |
| **Brainstorming** | 1.0-1.5 | 50-100 | 0.95-1.0 | 1.2-1.5 | 50-150 | Maximum creativity and diversity |
| **Summarization** | 0.3-0.5 | 20-40 | 0.7-0.8 | 1.0-1.2 | 50-150 | Focused, accurate, concise |
| **Long-form content** | 0.6-0.9 | 40-60 | 0.85-0.95 | 1.2-1.5 | 300-1000 | Balance creativity & coherence, avoid repetition |


## Part 6b: Best Practices for Parameter Selection

###  The Parameter Selection Process

**Step 1: Define Your Use Case**
- What type of content are you generating?
- Who is the audience?
- What's the priority: accuracy, creativity, or balance?

**Step 2: Start with Recommended Baseline**
- Use the quick reference table above
- Choose a configuration that matches your use case

**Step 3: Run A/B Tests**
- Test 3-5 variations around the baseline
- Generate multiple samples per configuration
- Use consistent prompts for fair comparison

**Step 4: Evaluate Systematically**
- Rate outputs on: relevance, coherence, creativity, accuracy
- Look for patterns across samples
- Consider edge cases

**Step 5: Fine-tune and Validate**
- Adjust parameters based on evaluation
- Test with diverse prompts
- Document your final configuration

### Common Parameter Selection Mistakes

1. **Testing only one sample per config** → Always generate multiple samples
2. **Changing all parameters at once** → Isolate variables when testing
3. **Ignoring max_length effects** → Longer outputs need repetition penalty adjustment
4. **Not testing edge cases** → Test with unusual or challenging prompts
5. **Overfitting to one prompt** → Validate across diverse inputs
6. **Extreme parameter values** → Very high/low values often produce poor results


## Part 7: Interactive Experimentation

Now it's your turn! Use this interactive cell to experiment with different parameters.

In [24]:
# YOUR EXPERIMENT ZONE 
# Modify these parameters and see how the output changes!

# Your custom prompt
my_prompt = "The most important skill for future professionals is"

# Your custom parameters - EXPERIMENT WITH THESE!
my_temperature = 0.7       # Try values from 0.1 to 2.0
my_top_k = 50             # Try values from 5 to 100
my_top_p = 0.9            # Try values from 0.5 to 1.0
my_repetition_penalty = 1.1  # Try values from 1.0 to 2.0
my_max_length = 80        # Try values from 30 to 200

print("=" * 80)
print(" YOUR CUSTOM EXPERIMENT")
print("=" * 80)
print(f"\nPrompt: '{my_prompt}'")
print(f"\nParameters:")
print(f"   Temperature: {my_temperature}")
print(f"   Top-k: {my_top_k}")
print(f"   Top-p: {my_top_p}")
print(f"   Repetition Penalty: {my_repetition_penalty}")
print(f"   Max Length: {my_max_length}\n")

my_results = generate_text(
    my_prompt,
    temperature=my_temperature,
    top_k=my_top_k,
    top_p=my_top_p,
    repetition_penalty=my_repetition_penalty,
    max_length=my_max_length,
    num_return_sequences=3
)

for i, text in enumerate(my_results, 1):
    print(f"{'─' * 80}")
    print(f"Output {i}:")
    print(f"{'─' * 80}")
    print(text)
    print()

print("=" * 80)
print(" Try different combinations and observe how they affect:")
print("   • Creativity vs. consistency")
print("   • Vocabulary diversity")
print("   • Output coherence")
print("   • Repetition (especially in longer outputs)")
print("=" * 80)

 YOUR CUSTOM EXPERIMENT

Prompt: 'The most important skill for future professionals is'

Parameters:
   Temperature: 0.7
   Top-k: 50
   Top-p: 0.9
   Repetition Penalty: 1.1
   Max Length: 80

────────────────────────────────────────────────────────────────────────────────
Output 1:
────────────────────────────────────────────────────────────────────────────────
The most important skill for future professionals is to be able, by now, to communicate effectively.
I think that you need a great understanding of the way people interact with each other and how they act as friends or partners in their lives. I would also like to ask what kind an organization can do about this problem: What are its best practices? How does one deal with it when there's no

────────────────────────────────────────────────────────────────────────────────
Output 2:
────────────────────────────────────────────────────────────────────────────────
The most important skill for future professionals is to have a clear

## Part 8: Understanding the Trade-offs

### Creativity vs. Consistency Spectrum

```
Deterministic          Balanced              Creative
(Predictable)         (Moderate)          (Exploratory)
     ◄──────────────────────────────────────────────►
     
Temp:       0.0-0.3         0.5-0.8              1.0-2.0
Top-k:      5-20            30-50                50-100
Top-p:      0.5-0.7         0.75-0.85            0.9-1.0
Rep. Pen:   1.0             1.0-1.2              1.2-1.5

Best for:             Best for:            Best for:
• Code                • General content    • Creative writing
• Technical docs      • Documentation      • Brainstorming
• Support             • Summaries          • Marketing copy
```

### Parameter Interactions

**Key Insight:** Parameters don't work in isolation—they interact with each other!

**Temperature + Top-p:**
- Low temp + Low top-p = Maximum determinism (technical, code)
- Low temp + High top-p = Consistent but varied vocabulary
- High temp + Low top-p = Creative within constraints
- High temp + High top-p = Maximum exploration (creative writing)

**Max Length + Repetition Penalty:**
- Short outputs (< 50 tokens) → Rep. penalty ~1.0 usually sufficient
- Medium outputs (50-150 tokens) → Rep. penalty 1.1-1.2 helps
- Long outputs (150+ tokens) → Rep. penalty 1.2-1.5 often necessary

**Top-k + Top-p:**
- Both limit vocabulary, but differently
- Top-k = fixed number, Top-p = dynamic based on distribution
- Common practice: Use top-p alone or combine conservatively

### Common Pitfalls to Avoid

1. **Too high temperature (>1.5):** Output becomes incoherent and random
2. **Too low temperature (<0.1):** Output becomes repetitive and boring  
3. **Very low top-k (<10):** Severely limits vocabulary, creates repetitive text
4. **Conflicting parameters:** High temperature + very low top-p can create unpredictable results
5. **Ignoring repetition penalty:** Long outputs without it become very repetitive
6. **Very high repetition penalty (>1.8):** Makes text unnatural and forced

### When Parameters Don't Work As Expected

**If outputs are too repetitive:**
- ✓ Increase temperature
- ✓ Increase top-p
- ✓ Increase repetition penalty
- ✓ Check if max_length is too long for your parameters

**If outputs are too random/incoherent:**
- ✓ Decrease temperature
- ✓ Decrease top-p
- ✓ Decrease top-k
- ✓ Check if temperature is above 1.2

**If outputs lack creativity:**
- ✓ Increase temperature (0.7-1.0)
- ✓ Increase top-p (0.9-0.95)
- ✓ Increase top-k (50-80)

**If outputs are off-topic:**
- ✓ Lower temperature
- ✓ Lower top-p
- ✓ Improve your prompt (this is often the real issue!)


In [22]:
# Demonstration: The pitfalls!
prompt = "Innovation in technology requires"

print("=" * 80)
print("  COMMON PITFALLS DEMONSTRATION")
print("=" * 80)

# Pitfall 1: Temperature too high
print("\n PITFALL 1: Temperature too high (2.0)")
print("Result: Incoherent, random text\n")
results = generate_text(prompt, temperature=2.0, top_k=50, top_p=0.9, max_length=60, num_return_sequences=1)
print(results[0])

# Pitfall 2: Top-k too low
print("\n" + "─" * 80)
print(" PITFALL 2: Top-k too low (5)")
print("Result: Repetitive, limited vocabulary\n")
results = generate_text(prompt, temperature=0.7, top_k=5, top_p=0.9, max_length=60, num_return_sequences=1)
print(results[0])

# Good balance
print("\n" + "─" * 80)
print(" OPTIMAL: Balanced parameters")
print("Parameters: Temp=0.7, Top-k=50, Top-p=0.9")
print("Result: Coherent, creative, and useful\n")
results = generate_text(prompt, temperature=0.7, top_k=50, top_p=0.9, max_length=60, num_return_sequences=1)
print(results[0])

print("\n" + "=" * 80)

  COMMON PITFALLS DEMONSTRATION

 PITFALL 1: Temperature too high (2.0)
Result: Incoherent, random text

Innovation in technology requires its customers to believe that its product meets consumers' needs. Unfortunately it will not deliver innovation. I do worry not because, with its unique business model, IBM has done nothing to reduce our commitment to innovation for years."


"What should a company achieve with the

────────────────────────────────────────────────────────────────────────────────
 PITFALL 2: Top-k too low (5)
Result: Repetitive, limited vocabulary

Innovation in technology requires a high level of technical knowledge and understanding of the business, and that knowledge must be shared with the team. This is especially true of a team of developers.

"The team is very focused on the business, and we have to understand the business and how it

────────────────────────────────────────────────────────────────────────────────
 OPTIMAL: Balanced parameters
Parameters: Temp=0

## Part 9: Key Takeaways and Best Practices

###  What You've Learned

1. **Temperature** controls the randomness of outputs
   - Low (0-0.3) = deterministic and consistent
   - Medium (0.7-1.0) = balanced
   - High (1.0-2.0) = creative and exploratory
   
2. **Top-k** limits vocabulary to k most likely tokens
   - Fixed constraint on token selection
   - Small k = focused, large k = diverse
   
3. **Top-p (nucleus sampling)** dynamically selects tokens by cumulative probability
   - Adaptive constraint that adjusts to context
   - More flexible than top-k
   
4. **Repetition penalty** reduces repeated words and phrases
   - Essential for longer text generation
   - Balance diversity with natural language flow
   
5. **Max length** controls output size
   - Consider task requirements
   - Adjust repetition penalty for longer outputs

###  A/B Testing Framework

**You've learned how to:**
- Systematically explore parameter space
- Design controlled experiments
- Compare outputs objectively
- Select optimal configurations for different use cases

### Decision Framework

**When choosing parameters, ask yourself:**

1. **How creative should the output be?**
   - Need creativity → Higher temperature (0.8-1.2), higher top-p (0.9+)
   - Need consistency → Lower temperature (0.2-0.5), lower top-p (0.6-0.8)

2. **How important is factual accuracy?**
   - Very important → Conservative parameters across the board
   - Less critical → Exploratory parameters for creativity

3. **How long is the output?**
   - Short outputs (< 100 tokens) → Repetition penalty ~1.0
   - Long outputs (> 200 tokens) → Repetition penalty 1.2-1.5

4. **What's the application context?**
   - Technical/Support → Conservative (temp 0.2-0.5)
   - Creative/Marketing → Exploratory (temp 0.8-1.2)
   - Balanced/General → Middle ground (temp 0.6-0.8)

### Next Steps

- **Practice:** Experiment with different prompts and parameter combinations
- **Test:** Apply A/B testing on your specific use cases
- **Measure:** Use evaluation metrics (covered in next module!)
- **Iterate:** Refine parameters based on results
- **Document:** Keep track of what works for different scenarios

### Advanced Topics to Explore

- **Evaluation metrics** for systematic quality assessment (Module 1, Video 4)
- **Contrastive search** for reducing repetition
- **Diverse beam search** for multiple high-quality outputs
- **Fine-tuning** for task-specific improvements (Module 2 & 3)


In [23]:
# 📝 YOUR ASSIGNMENT WORKSPACE
# Use this cell to complete the practice assignment

# TODO: Test parameters for creative blog post
# TODO: Test parameters for technical summary
# TODO: Test parameters for customer support

# Example starter code:
print("Assignment Workspace - Complete the three scenarios above!\n")

# Scenario 1: Creative blog post
# Your code here...

# Scenario 2: Technical summary  
# Your code here...

# Scenario 3: Customer support
# Your code here...

Assignment Workspace - Complete the three scenarios above!



## Additional Resources

**Documentation:**
- [Hugging Face Text Generation Strategies](https://huggingface.co/docs/transformers/generation_strategies)
- [The Curious Case of Neural Text Degeneration (Top-p paper)](https://arxiv.org/abs/1904.09751)
- [Temperature Sampling in Language Models](https://arxiv.org/abs/1904.09751)



---

## Congratulations!

You've completed **Module 1, Video 3: Tuning Decoding Parameters in Practice**!

You now understand:
- How language models generate text from probabilities
- The role of temperature, top-k, and top-p in controlling outputs
- How to balance creativity and consistency for different applications
- Best practices for parameter tuning in real-world scenarios

**Ready to move forward?** Continue to the next video to learn about evaluation metrics!

---